# Selective relation fallback on 17 VALID images

Attach csa-scst-full-karpathy-cache-v1, csa-prompt-full-karpathy-v175, upt-prompts-full-all-actions-v2 and mscoco-2014. Enable Internet, Secret codex and GPU T4. Exact SCST1500 is fetched from v94. Send csa_relation_fallback_caption_pilot_review.zip.

In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/94'
SECRET_NAME='codex'
RESUME_SHA='80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24'
WALL_HOURS=3 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_relation_fallback_caption_pilot')
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}
CACHE_ROOT='/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1'
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)


PROPOSALS=[{'case_id': 1, 'coco_id': 122542, 'filename': 'COCO_val2014_000000122542.jpg', 'text': 'a person standing on skis', 'subject_roi': [0.11083999633789063, 0.2975375430362003, 0.24894000244140624, 0.8497898227817662], 'upt_object': [0.18878441905101867, 0.7772064208984375, 0.4084931055175375, 0.9135649108886719], 'yolo_object': [0.2298800048828125, 0.8189789826447541, 0.39933999633789063, 0.9014414194467906], 'object_roi': 2, 'iou': 0.4664380096437739, 'predicate': 'stand_on', 'object_instance': 'det_144dfc0835ff'}, {'case_id': 2, 'coco_id': 25005, 'filename': 'COCO_val2014_000000025005.jpg', 'text': 'a person wearing skis', 'subject_roi': [0.07140515168880132, 0.10479687452316284, 0.8845667247191525, 0.8936093330383301], 'upt_object': [0.35732078552246094, 0.863229835898405, 0.9918666839599609, 0.9108060020720392], 'yolo_object': [0.3631381675845287, 0.8786874771118164, 0.9970491574575527, 0.9004843711853028], 'object_roi': 3, 'iou': 0.45225490228673815, 'predicate': 'wear', 'object_instance': 'det_23e07b93f9a7'}, {'case_id': 3, 'coco_id': 25005, 'filename': 'COCO_val2014_000000025005.jpg', 'text': 'a person standing on skis', 'subject_roi': [0.07140515168880132, 0.10479687452316284, 0.8845667247191525, 0.8936093330383301], 'upt_object': [0.35732078552246094, 0.863229835898405, 0.9918666839599609, 0.9108060020720392], 'yolo_object': [0.3631381675845287, 0.8786874771118164, 0.9970491574575527, 0.9004843711853028], 'object_roi': 3, 'iou': 0.45225490228673815, 'predicate': 'stand_on', 'object_instance': 'det_23e07b93f9a7'}, {'case_id': 4, 'coco_id': 528276, 'filename': 'COCO_val2014_000000528276.jpg', 'text': 'a person riding a skateboard', 'subject_roi': [0.4466562271118164, 0.42374707831711067, 0.5098437309265137, 0.6044964756842799], 'upt_object': [0.447643340479045, 0.579435043334961, 0.5115289250645069, 0.611900520324707], 'yolo_object': [0.45098438262939455, 0.588501155236845, 0.5077812671661377, 0.6067446992324722], 'object_roi': 1, 'iou': 0.4995846216256837, 'predicate': 'riding', 'object_instance': 'det_42d34fd955e5'}, {'case_id': 5, 'coco_id': 528276, 'filename': 'COCO_val2014_000000528276.jpg', 'text': 'a person standing on a skateboard', 'subject_roi': [0.4466562271118164, 0.42374707831711067, 0.5098437309265137, 0.6044964756842799], 'upt_object': [0.447643340479045, 0.579435043334961, 0.5115289250645069, 0.611900520324707], 'yolo_object': [0.45098438262939455, 0.588501155236845, 0.5077812671661377, 0.6067446992324722], 'object_roi': 1, 'iou': 0.4995846216256837, 'predicate': 'stand_on', 'object_instance': 'det_42d34fd955e5'}, {'case_id': 6, 'coco_id': 80895, 'filename': 'COCO_val2014_000000080895.jpg', 'text': 'a person reading a cell phone', 'subject_roi': [0.2789843797683716, 0.005958333114782969, 0.9984531402587891, 0.989437484741211], 'upt_object': [0.3227850822749326, 0.5727752304077148, 0.4334103582500293, 0.8142823028564453], 'yolo_object': [0.3194531202316284, 0.6810000101725261, 0.43704686164855955, 0.8032916386922201], 'object_roi': 2, 'iou': 0.4907163526688554, 'predicate': 'read', 'object_instance': 'det_213caa1e50f1'}, {'case_id': 7, 'coco_id': 80895, 'filename': 'COCO_val2014_000000080895.jpg', 'text': 'a person holding a cell phone', 'subject_roi': [0.2789843797683716, 0.005958333114782969, 0.9984531402587891, 0.989437484741211], 'upt_object': [0.3227850822749326, 0.5727752304077148, 0.4334103582500293, 0.8142823028564453], 'yolo_object': [0.3194531202316284, 0.6810000101725261, 0.43704686164855955, 0.8032916386922201], 'object_roi': 2, 'iou': 0.4907163526688554, 'predicate': 'holding', 'object_instance': 'det_213caa1e50f1'}, {'case_id': 8, 'coco_id': 336431, 'filename': 'COCO_val2014_000000336431.jpg', 'text': 'a person holding a toothbrush', 'subject_roi': [0.10564732551574707, 0.10807812213897705, 0.9012276785714286, 0.9904999732971191], 'upt_object': [0.18619701385498047, 0.10627414549712333, 0.34645469665527345, 0.437118770779745], 'yolo_object': [0.25430803639548166, 0.10932812690734864, 0.3379241057804653, 0.42618751525878906], 'object_roi': 2, 'iou': 0.4997046437267972, 'predicate': 'holding', 'object_instance': 'det_b468a0f6162a'}, {'case_id': 9, 'coco_id': 48554, 'filename': 'COCO_val2014_000000048554.jpg', 'text': 'a person riding a skateboard', 'subject_roi': [0.24497999572753906, 0.6193993829034112, 0.293, 0.7670870944186374], 'upt_object': [0.2389288020074417, 0.7423151397705078, 0.30412870561153466, 0.7773525238037109], 'yolo_object': [0.24302000427246093, 0.7547447502434075, 0.29779998779296873, 0.7740540833802553], 'object_roi': 8, 'iou': 0.46303150913969465, 'predicate': 'riding', 'object_instance': 'det_7f57fa797f59'}, {'case_id': 10, 'coco_id': 366822, 'filename': 'COCO_val2014_000000366822.jpg', 'text': 'a person holding a snowboard', 'subject_roi': [0.10280000305175781, 0.2394666748046875, 0.4184400024414062, 0.984106689453125], 'upt_object': [0.17442721139646605, 0.497357177734375, 0.3944267000982059, 0.9937321472167969], 'yolo_object': [0.18547999572753907, 0.504933349609375, 0.378739990234375, 0.7838400065104166], 'object_roi': 5, 'iou': 0.49359334782980324, 'predicate': 'holding', 'object_instance': 'det_996a84230a99'}, {'case_id': 11, 'coco_id': 135155, 'filename': 'COCO_val2014_000000135155.jpg', 'text': 'a person wearing skis', 'subject_roi': [0.24198436737060547, 0.5503333409627279, 0.2982968807220459, 0.7495625178019206], 'upt_object': [0.24347825613970156, 0.7067706298828125, 0.2982381280322907, 0.7617987823486329], 'yolo_object': [0.2506718635559082, 0.7242708206176758, 0.2935312509536743, 0.756624984741211], 'object_roi': 2, 'iou': 0.4601811108294064, 'predicate': 'wear', 'object_instance': 'det_9cfe365b73f8'}, {'case_id': 12, 'coco_id': 135155, 'filename': 'COCO_val2014_000000135155.jpg', 'text': 'a person standing on skis', 'subject_roi': [0.24198436737060547, 0.5503333409627279, 0.2982968807220459, 0.7495625178019206], 'upt_object': [0.24347825613970156, 0.7067706298828125, 0.2982381280322907, 0.7617987823486329], 'yolo_object': [0.2506718635559082, 0.7242708206176758, 0.2935312509536743, 0.756624984741211], 'object_roi': 2, 'iou': 0.4601811108294064, 'predicate': 'stand_on', 'object_instance': 'det_9cfe365b73f8'}, {'case_id': 13, 'coco_id': 135155, 'filename': 'COCO_val2014_000000135155.jpg', 'text': 'a person on skis', 'subject_roi': [0.24198436737060547, 0.5503333409627279, 0.2982968807220459, 0.7495625178019206], 'upt_object': [0.24347825613970156, 0.7067706298828125, 0.2982381280322907, 0.7617987823486329], 'yolo_object': [0.2506718635559082, 0.7242708206176758, 0.2935312509536743, 0.756624984741211], 'object_roi': 2, 'iou': 0.4601811108294064, 'predicate': 'riding', 'object_instance': 'det_9cfe365b73f8'}, {'case_id': 14, 'coco_id': 217622, 'filename': 'COCO_val2014_000000217622.jpg', 'text': 'a person riding a surfboard', 'subject_roi': [0.49721875190734866, 0.4014117790670956, 0.6520156383514404, 0.6470823759191177], 'upt_object': [0.38414143407067586, 0.6071509552001954, 0.5678990259519051, 0.7169354248046875], 'yolo_object': [0.3900781154632568, 0.6401647231158089, 0.5268437385559082, 0.7078352625229779], 'object_roi': 1, 'iou': 0.4587651166601786, 'predicate': 'riding', 'object_instance': 'det_ec1d734e05e5'}, {'case_id': 15, 'coco_id': 217622, 'filename': 'COCO_val2014_000000217622.jpg', 'text': 'a person standing on a surfboard', 'subject_roi': [0.49721875190734866, 0.4014117790670956, 0.6520156383514404, 0.6470823759191177], 'upt_object': [0.38414143407067586, 0.6071509552001954, 0.5678990259519051, 0.7169354248046875], 'yolo_object': [0.3900781154632568, 0.6401647231158089, 0.5268437385559082, 0.7078352625229779], 'object_roi': 1, 'iou': 0.4587651166601786, 'predicate': 'stand_on', 'object_instance': 'det_ec1d734e05e5'}, {'case_id': 16, 'coco_id': 15303, 'filename': 'COCO_val2014_000000015303.jpg', 'text': 'a person riding a surfboard', 'subject_roi': [0.3430156230926514, 0.07981481501665065, 0.6162343978881836, 0.4853968140940187], 'upt_object': [0.24849180845884947, 0.12514110986886515, 0.7213850050001173, 0.5332218543395608], 'yolo_object': [0.25679688453674315, 0.140343913325557, 0.4888750076293945, 0.5171428458400504], 'object_roi': 1, 'iou': 0.4531423979250723, 'predicate': 'riding', 'object_instance': 'det_8c3c806398b9'}, {'case_id': 17, 'coco_id': 15303, 'filename': 'COCO_val2014_000000015303.jpg', 'text': 'a person standing on a surfboard', 'subject_roi': [0.3430156230926514, 0.07981481501665065, 0.6162343978881836, 0.4853968140940187], 'upt_object': [0.24849180845884947, 0.12514110986886515, 0.7213850050001173, 0.5332218543395608], 'yolo_object': [0.25679688453674315, 0.140343913325557, 0.4888750076293945, 0.5171428458400504], 'object_roi': 1, 'iou': 0.4531423979250723, 'predicate': 'stand_on', 'object_instance': 'det_8c3c806398b9'}, {'case_id': 18, 'coco_id': 294537, 'filename': 'COCO_val2014_000000294537.jpg', 'text': 'a person standing on skis', 'subject_roi': [0.6600163279016034, 0.3567156822852839, 0.7906045851364635, 0.6740849812825521], 'upt_object': [0.6648792266845703, 0.6503560638427734, 0.8082583618164062, 0.6794855499267578], 'yolo_object': [0.6729738322737949, 0.6641340068742341, 0.8082516240138634, 0.6802450940499898], 'object_roi': 6, 'iou': 0.4852938574014861, 'predicate': 'stand_on', 'object_instance': 'det_3b75773100c5'}, {'case_id': 19, 'coco_id': 73927, 'filename': 'COCO_val2014_000000073927.jpg', 'text': 'a person wielding a baseball bat', 'subject_roi': [0.29689061641693115, 0.08419928771321035, 0.4328437328338623, 0.5433451696647019], 'upt_object': [0.22514622777073948, 0.11282872909154647, 0.3213210922103744, 0.15677779760116187], 'yolo_object': [0.23212499618530275, 0.12145907853421792, 0.3069062471389771, 0.14814946999329265], 'object_roi': 4, 'iou': 0.472211339394478, 'predicate': 'wield', 'object_instance': 'det_13178f55a325'}, {'case_id': 20, 'coco_id': 557596, 'filename': 'COCO_val2014_000000557596.jpg', 'text': 'a person holding a tennis racket', 'subject_roi': [0.20867611499542885, 0.1059999942779541, 0.8394090007664746, 0.8384843826293945], 'upt_object': [0.1826953887939453, 0.0019924268249637824, 0.37513137817382813, 0.21899378752905474], 'yolo_object': [0.20144207821388335, 0.0006250000093132258, 0.371702117559194, 0.12146874666213989], 'object_roi': 1, 'iou': 0.48443020673348536, 'predicate': 'holding', 'object_instance': 'det_791a8dcda111'}, {'case_id': 21, 'coco_id': 557596, 'filename': 'COCO_val2014_000000557596.jpg', 'text': 'a person swinging a tennis racket', 'subject_roi': [0.20867611499542885, 0.1059999942779541, 0.8394090007664746, 0.8384843826293945], 'upt_object': [0.1826953887939453, 0.0019924268249637824, 0.37513137817382813, 0.21899378752905474], 'yolo_object': [0.20144207821388335, 0.0006250000093132258, 0.371702117559194, 0.12146874666213989], 'object_roi': 1, 'iou': 0.48443020673348536, 'predicate': 'swing', 'object_instance': 'det_791a8dcda111'}, {'case_id': 22, 'coco_id': 557596, 'filename': 'COCO_val2014_000000557596.jpg', 'text': 'a person carrying a tennis racket', 'subject_roi': [0.20867611499542885, 0.1059999942779541, 0.8394090007664746, 0.8384843826293945], 'upt_object': [0.1826953887939453, 0.0019924268249637824, 0.37513137817382813, 0.21899378752905474], 'yolo_object': [0.20144207821388335, 0.0006250000093132258, 0.371702117559194, 0.12146874666213989], 'object_roi': 1, 'iou': 0.48443020673348536, 'predicate': 'carry', 'object_instance': 'det_791a8dcda111'}, {'case_id': 23, 'coco_id': 29573, 'filename': 'COCO_val2014_000000029573.jpg', 'text': 'a person wearing a tie', 'subject_roi': [0.11227165954732783, 0.016781249642372133, 0.9212646484375, 0.9995937347412109], 'upt_object': [0.3052156257629395, 0.1722833857723233, 0.36412139892578127, 0.30072869053475554], 'yolo_object': [0.3138641786240303, 0.17942187786102295, 0.34658080911748024, 0.3175156354904175], 'object_roi': 1, 'iou': 0.4890404467406889, 'predicate': 'wear', 'object_instance': 'det_1cc5319b5533'}, {'case_id': 24, 'coco_id': 131869, 'filename': 'COCO_val2014_000000131869.jpg', 'text': 'a person holding a tennis racket', 'subject_roi': [0.37256001790364585, 0.36207998657226564, 0.5568800048828125, 0.580260009765625], 'upt_object': [0.5389320755004883, 0.3366272373449959, 0.645422592163086, 0.40266532969519525], 'yolo_object': [0.5575733235677083, 0.34682000732421875, 0.63517333984375, 0.3908599853515625], 'object_roi': 3, 'iou': 0.4859632776835211, 'predicate': 'holding', 'object_instance': 'det_a2fd79bb295e'}, {'case_id': 25, 'coco_id': 131869, 'filename': 'COCO_val2014_000000131869.jpg', 'text': 'a person carrying a tennis racket', 'subject_roi': [0.37256001790364585, 0.36207998657226564, 0.5568800048828125, 0.580260009765625], 'upt_object': [0.5389320755004883, 0.3366272373449959, 0.645422592163086, 0.40266532969519525], 'yolo_object': [0.5575733235677083, 0.34682000732421875, 0.63517333984375, 0.3908599853515625], 'object_roi': 3, 'iou': 0.4859632776835211, 'predicate': 'carry', 'object_instance': 'det_a2fd79bb295e'}, {'case_id': 26, 'coco_id': 131869, 'filename': 'COCO_val2014_000000131869.jpg', 'text': 'a person swinging a tennis racket', 'subject_roi': [0.37256001790364585, 0.36207998657226564, 0.5568800048828125, 0.580260009765625], 'upt_object': [0.5389320755004883, 0.3366272373449959, 0.645422592163086, 0.40266532969519525], 'yolo_object': [0.5575733235677083, 0.34682000732421875, 0.63517333984375, 0.3908599853515625], 'object_roi': 3, 'iou': 0.4859632776835211, 'predicate': 'swing', 'object_instance': 'det_a2fd79bb295e'}, {'case_id': 27, 'coco_id': 30347, 'filename': 'COCO_val2014_000000030347.jpg', 'text': 'a person reading a laptop', 'subject_roi': [0.33767999267578125, 0.06202666727701823, 0.7047999877929687, 0.39909334309895833], 'upt_object': [0.5591247444081262, 0.2494676399230957, 0.6684231355534709, 0.47418941497802736], 'yolo_object': [0.5750399780273437, 0.25002667236328124, 0.74497998046875, 0.46072001139322916], 'object_roi': 9, 'iou': 0.4835184361319553, 'predicate': 'read', 'object_instance': 'det_9e266df88e55'}, {'case_id': 28, 'coco_id': 4375, 'filename': 'COCO_val2014_000000004375.jpg', 'text': 'a person standing on skis', 'subject_roi': [0.09051999664306641, 0.6816816816816816, 0.1280800018310547, 0.8420120560013138], 'upt_object': [0.057249559947195695, 0.8205406951904297, 0.15579083877042568, 0.8533350372314453], 'yolo_object': [0.06206000137329101, 0.8312012268616272, 0.1563000030517578, 0.8478078298024587], 'object_roi': 13, 'iou': 0.4804092193208496, 'predicate': 'stand_on', 'object_instance': 'det_dfaaade63641'}, {'case_id': 29, 'coco_id': 4375, 'filename': 'COCO_val2014_000000004375.jpg', 'text': 'a person wearing skis', 'subject_roi': [0.09051999664306641, 0.6816816816816816, 0.1280800018310547, 0.8420120560013138], 'upt_object': [0.057249559947195695, 0.8205406951904297, 0.15579083877042568, 0.8533350372314453], 'yolo_object': [0.06206000137329101, 0.8312012268616272, 0.1563000030517578, 0.8478078298024587], 'object_roi': 13, 'iou': 0.4804092193208496, 'predicate': 'wear', 'object_instance': 'det_dfaaade63641'}]
SAVED={4375: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000004375.jpg', 'image_sha256': '999c2ccf98b2798b57175a0e75d72fe856b5d05d8d1f8d3f992ec79f04a2d71d', 'detected': {'person': 0.7844}, 'selected': 2, 'beams': [{'ids': [1, 4, 34, 6, 19, 18, 5, 33, 6, 4, 74, 132, 234, 2, 0, 0, 0], 'caption': 'a group of people standing on top of a snow covered slope', 'logprob': -4.346950054168701}, {'ids': [1, 4, 34, 6, 19, 47, 157, 5, 33, 6, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a group of people riding skis on top of a snow covered slope', 'logprob': -4.375782489776611}, {'ids': [1, 4, 34, 6, 19, 47, 157, 5, 4, 74, 132, 234, 2, 0, 0, 0, 0], 'caption': 'a group of people riding skis on a snow covered slope', 'logprob': -4.675085544586182}, {'ids': [1, 4, 34, 6, 19, 47, 157, 5, 4, 247, 568, 2, 0, 0, 0, 0, 0], 'caption': 'a group of people riding skis on a snowy surface', 'logprob': -5.3036603927612305}, {'ids': [1, 4, 34, 6, 19, 47, 157, 32, 4, 74, 132, 234, 2, 0, 0, 0, 0], 'caption': 'a group of people riding skis down a snow covered slope', 'logprob': -5.344404697418213}]}, 15303: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000015303.jpg', 'image_sha256': '0937e550afd880c8478e0b3667317ffffaae53329e8b3be532b7e6ebd559c9d9', 'detected': {'person': 0.8874, 'surfboard': 0.5934}, 'selected': 0, 'beams': [{'ids': [1, 4, 12, 47, 4, 194, 5, 33, 6, 4, 155, 2, 0, 0], 'caption': 'a man riding a wave on top of a surfboard', 'logprob': -2.3176019191741943}, {'ids': [1, 4, 12, 5, 4, 155, 47, 4, 194, 2, 0, 0, 0, 0], 'caption': 'a man on a surfboard riding a wave', 'logprob': -3.2225353717803955}, {'ids': [1, 4, 12, 47, 4, 194, 5, 4, 155, 2, 0, 0, 0, 0], 'caption': 'a man riding a wave on a surfboard', 'logprob': -3.736757755279541}, {'ids': [1, 4, 31, 5, 4, 155, 47, 4, 194, 2, 0, 0, 0, 0], 'caption': 'a person on a surfboard riding a wave', 'logprob': -3.7978098392486572}, {'ids': [1, 4, 12, 47, 4, 155, 5, 33, 6, 4, 194, 2, 0, 0], 'caption': 'a man riding a surfboard on top of a wave', 'logprob': -4.134450912475586}]}, 25005: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000025005.jpg', 'image_sha256': '4503c81279480d339c13f79d64a63609fc50455605a3956d332965c140a80658', 'detected': {'person': 0.9408, 'skis': 0.7336}, 'selected': 1, 'beams': [{'ids': [1, 4, 12, 47, 157, 32, 4, 74, 132, 234, 2, 0, 0, 0], 'caption': 'a man riding skis down a snow covered slope', 'logprob': -2.829281806945801}, {'ids': [1, 4, 12, 47, 157, 241, 74, 132, 209, 2, 0, 0, 0, 0], 'caption': 'a man riding skis across snow covered ground', 'logprob': -3.0253591537475586}, {'ids': [1, 4, 31, 5, 30, 157, 8, 7, 74, 2, 0, 0, 0, 0], 'caption': 'a person on some skis in the snow', 'logprob': -3.889937162399292}, {'ids': [1, 4, 12, 5, 30, 157, 8, 7, 74, 2, 0, 0, 0, 0], 'caption': 'a man on some skis in the snow', 'logprob': -4.01954984664917}, {'ids': [1, 4, 12, 47, 157, 5, 33, 6, 74, 132, 209, 2, 0, 0], 'caption': 'a man riding skis on top of snow covered ground', 'logprob': -4.314706325531006}]}, 29573: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000029573.jpg', 'image_sha256': '7da3cca824b4cddd6212479479d4e62ed03329e9267096aab3d211b4a7604f29', 'detected': {'person': 0.9231, 'tie': 0.5954}, 'selected': 0, 'beams': [{'ids': [1, 4, 12, 8, 4, 263, 57, 8, 41, 6, 4, 70, 2, 0, 0, 0], 'caption': 'a man in a suit walking in front of a building', 'logprob': -6.454921722412109}, {'ids': [1, 4, 12, 8, 4, 263, 57, 32, 4, 70, 2, 0, 0, 0, 0, 0], 'caption': 'a man in a suit walking down a building', 'logprob': -6.781100273132324}, {'ids': [1, 4, 12, 8, 4, 263, 10, 211, 57, 8, 41, 6, 4, 70, 2, 0], 'caption': 'a man in a suit and tie walking in front of a building', 'logprob': -6.96556282043457}, {'ids': [1, 4, 12, 8, 4, 263, 57, 8, 41, 6, 4, 22, 70, 2, 0, 0], 'caption': 'a man in a suit walking in front of a white building', 'logprob': -7.755585670471191}, {'ids': [1, 4, 12, 8, 4, 263, 10, 211, 57, 32, 4, 70, 2, 0, 0, 0], 'caption': 'a man in a suit and tie walking down a building', 'logprob': -7.920975208282471}]}, 30347: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000030347.jpg', 'image_sha256': '1136e04bdc8a729ba237ee8058ee5678bf4ade1d80ab134285d987b3b9e9b5d4', 'detected': {'person': 0.9523, 'laptop': 0.8665, 'couch': 0.6988, 'chair': 0.6964, 'remote': 0.5425, 'refrigerator': 0.5131}, 'selected': 1, 'beams': [{'ids': [1, 4, 34, 6, 19, 14, 108, 4, 25, 9, 4, 104, 2, 0, 0, 0, 0, 0, 0], 'caption': 'a group of people sitting around a table with a laptop', 'logprob': -4.026438236236572}, {'ids': [1, 4, 34, 6, 19, 18, 108, 4, 104, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'caption': 'a group of people standing around a laptop', 'logprob': -4.59979772567749}, {'ids': [1, 4, 12, 18, 8, 41, 6, 4, 104, 115, 2, 0, 0, 0, 0, 0, 0, 0, 0], 'caption': 'a man standing in front of a laptop computer', 'logprob': -4.901297569274902}, {'ids': [1, 4, 34, 6, 19, 14, 8, 4, 123, 44, 2, 0, 0, 0, 0, 0, 0, 0, 0], 'caption': 'a group of people sitting in a living room', 'logprob': -5.178534507751465}, {'ids': [1, 4, 34, 6, 19, 14, 108, 4, 25, 8, 41, 6, 4, 104, 2, 0, 0, 0, 0], 'caption': 'a group of people sitting around a table in front of a laptop', 'logprob': -5.778901100158691}]}, 48554: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000048554.jpg', 'image_sha256': '859837b4332079677cd95f952be05bd1026488cb165772093427af0ee78d3c4a', 'detected': {'person': 0.8929, 'skateboard': 0.7524, 'sports ball': 0.5981}, 'selected': 0, 'beams': [{'ids': [1, 4, 34, 6, 406, 56, 4, 93, 6, 1348, 2, 0, 0, 0, 0, 0], 'caption': 'a group of kids playing a game of basketball', 'logprob': -3.9680593013763428}, {'ids': [1, 4, 34, 6, 19, 56, 4, 93, 6, 1348, 2, 0, 0, 0, 0, 0], 'caption': 'a group of people playing a game of basketball', 'logprob': -4.313266754150391}, {'ids': [1, 4, 34, 6, 50, 19, 56, 4, 93, 6, 1348, 2, 0, 0, 0, 0], 'caption': 'a group of young people playing a game of basketball', 'logprob': -4.519529342651367}, {'ids': [1, 4, 34, 6, 268, 56, 4, 93, 6, 1348, 2, 0, 0, 0, 0, 0], 'caption': 'a group of children playing a game of basketball', 'logprob': -4.6158599853515625}, {'ids': [1, 4, 34, 6, 50, 268, 56, 4, 93, 6, 1348, 2, 0, 0, 0, 0], 'caption': 'a group of young children playing a game of basketball', 'logprob': -5.309101104736328}]}, 73927: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000073927.jpg', 'image_sha256': '47a9ebf3343822f0a0b1cf985b54e0ba4de9cc67e07aa8a13f1e2467aac99d49', 'detected': {'person': 0.9364, 'baseball glove': 0.6945, 'baseball bat': 0.6306}, 'selected': 0, 'beams': [{'ids': [1, 4, 53, 89, 27, 4, 161, 5, 4, 36, 2, 0, 0, 0, 0], 'caption': 'a baseball player holding a bat on a field', 'logprob': -3.8921658992767334}, {'ids': [1, 4, 34, 6, 81, 5, 4, 36, 56, 53, 2, 0, 0, 0, 0], 'caption': 'a group of men on a field playing baseball', 'logprob': -4.293306350708008}, {'ids': [1, 4, 12, 27, 4, 53, 161, 5, 4, 36, 2, 0, 0, 0, 0], 'caption': 'a man holding a baseball bat on a field', 'logprob': -4.582573890686035}, {'ids': [1, 4, 53, 89, 27, 4, 53, 161, 5, 4, 36, 2, 0, 0, 0], 'caption': 'a baseball player holding a baseball bat on a field', 'logprob': -4.654587268829346}, {'ids': [1, 4, 34, 6, 81, 5, 4, 36, 56, 4, 93, 6, 53, 2, 0], 'caption': 'a group of men on a field playing a game of baseball', 'logprob': -4.7705841064453125}]}, 80895: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000080895.jpg', 'image_sha256': 'ccccd327374176e5b9867eb5502e306c17abb1c082e330dbefbbf3ded6e612db', 'detected': {'person': 0.9403, 'cell phone': 0.7348, 'boat': 0.6232}, 'selected': 0, 'beams': [{'ids': [1, 4, 12, 10, 4, 23, 72, 17, 106, 175, 611, 2, 0, 0], 'caption': 'a man and a woman looking at their cell phones', 'logprob': -3.1482741832733154}, {'ids': [1, 4, 12, 10, 4, 23, 20, 72, 17, 106, 175, 611, 2, 0], 'caption': 'a man and a woman are looking at their cell phones', 'logprob': -4.5111188888549805}, {'ids': [1, 4, 12, 10, 23, 72, 17, 106, 175, 611, 2, 0, 0, 0], 'caption': 'a man and woman looking at their cell phones', 'logprob': -4.5842108726501465}, {'ids': [1, 4, 12, 10, 4, 23, 72, 17, 106, 1748, 2, 0, 0, 0], 'caption': 'a man and a woman looking at their cellphones', 'logprob': -4.657793045043945}, {'ids': [1, 4, 12, 10, 4, 23, 72, 17, 4, 175, 116, 2, 0, 0], 'caption': 'a man and a woman looking at a cell phone', 'logprob': -4.822301387786865}]}, 122542: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000122542.jpg', 'image_sha256': '622083b6ed4285f7e32571d9bc660ecaef750412ec09a8f75ecef1c132412b46', 'detected': {'person': 0.9096, 'skis': 0.7387}, 'selected': 0, 'beams': [{'ids': [1, 4, 12, 10, 23, 18, 5, 157, 8, 7, 74, 2, 0, 0, 0], 'caption': 'a man and woman standing on skis in the snow', 'logprob': -4.786560535430908}, {'ids': [1, 4, 12, 10, 4, 23, 18, 5, 157, 8, 7, 74, 2, 0, 0], 'caption': 'a man and a woman standing on skis in the snow', 'logprob': -4.870030879974365}, {'ids': [1, 4, 12, 10, 23, 18, 8, 7, 74, 5, 157, 2, 0, 0, 0], 'caption': 'a man and woman standing in the snow on skis', 'logprob': -4.954612731933594}, {'ids': [1, 4, 12, 10, 4, 23, 18, 8, 7, 74, 5, 157, 2, 0, 0], 'caption': 'a man and a woman standing in the snow on skis', 'logprob': -5.206826210021973}, {'ids': [1, 4, 12, 10, 4, 23, 5, 157, 8, 7, 74, 2, 0, 0, 0], 'caption': 'a man and a woman on skis in the snow', 'logprob': -5.262457370758057}]}, 131869: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000131869.jpg', 'image_sha256': 'f8145242fc1fd0e9f64fc4abf31bdf3d84e464367968a2a093a22fdd2787d323', 'detected': {'person': 0.9086}, 'selected': 1, 'beams': [{'ids': [1, 4, 12, 27, 4, 38, 352, 5, 4, 38, 149, 2, 0, 0, 0], 'caption': 'a man holding a tennis racquet on a tennis court', 'logprob': -2.532683849334717}, {'ids': [1, 4, 12, 18, 5, 4, 38, 149, 27, 4, 352, 2, 0, 0, 0], 'caption': 'a man standing on a tennis court holding a racquet', 'logprob': -2.635516881942749}, {'ids': [1, 4, 12, 18, 5, 4, 38, 149, 27, 4, 38, 352, 2, 0, 0], 'caption': 'a man standing on a tennis court holding a tennis racquet', 'logprob': -3.409937858581543}, {'ids': [1, 4, 12, 5, 4, 38, 149, 27, 4, 38, 352, 2, 0, 0, 0], 'caption': 'a man on a tennis court holding a tennis racquet', 'logprob': -4.425405979156494}, {'ids': [1, 4, 12, 5, 4, 38, 149, 27, 4, 38, 207, 2, 0, 0, 0], 'caption': 'a man on a tennis court holding a tennis racket', 'logprob': -4.495731353759766}]}, 135155: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000135155.jpg', 'image_sha256': '077d6ea1f51c110ef8429f5dadeef45213a513fb7f2b9c4d5e808228a36a4399', 'detected': {'person': 0.904, 'skis': 0.5297}, 'selected': 1, 'beams': [{'ids': [1, 4, 34, 6, 19, 47, 157, 32, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a group of people riding skis down a snow covered slope', 'logprob': -3.5907106399536133}, {'ids': [1, 4, 77, 6, 19, 47, 157, 32, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a couple of people riding skis down a snow covered slope', 'logprob': -3.8110454082489014}, {'ids': [1, 4, 34, 6, 19, 47, 157, 241, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a group of people riding skis across a snow covered slope', 'logprob': -4.573523044586182}, {'ids': [1, 4, 77, 6, 19, 47, 157, 241, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a couple of people riding skis across a snow covered slope', 'logprob': -4.810800075531006}, {'ids': [1, 4, 34, 6, 19, 47, 157, 5, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a group of people riding skis on a snow covered slope', 'logprob': -5.450891017913818}]}, 217622: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000217622.jpg', 'image_sha256': 'ba17708fd73b1e4286c2a24fb9647ab97c5d5d44d568b5e9c5f710b45dc8d643', 'detected': {'person': 0.8888}, 'selected': 1, 'beams': [{'ids': [1, 4, 12, 47, 4, 194, 5, 33, 6, 4, 155, 2, 0, 0], 'caption': 'a man riding a wave on top of a surfboard', 'logprob': -2.692620038986206}, {'ids': [1, 4, 31, 5, 4, 155, 47, 4, 194, 2, 0, 0, 0, 0], 'caption': 'a person on a surfboard riding a wave', 'logprob': -3.583329677581787}, {'ids': [1, 4, 12, 5, 4, 155, 47, 4, 194, 2, 0, 0, 0, 0], 'caption': 'a man on a surfboard riding a wave', 'logprob': -3.644144296646118}, {'ids': [1, 4, 31, 47, 4, 194, 5, 33, 6, 4, 155, 2, 0, 0], 'caption': 'a person riding a wave on top of a surfboard', 'logprob': -3.826329231262207}, {'ids': [1, 4, 12, 47, 4, 194, 5, 4, 155, 2, 0, 0, 0, 0], 'caption': 'a man riding a wave on a surfboard', 'logprob': -4.305011749267578}]}, 294537: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000294537.jpg', 'image_sha256': '1863125b46e51a5222f66976779315c8b9856f465f08fc81903c6983cc17dbc7', 'detected': {'person': 0.9299, 'skis': 0.7583, 'dog': 0.6077, 'backpack': 0.587}, 'selected': 3, 'beams': [{'ids': [1, 4, 34, 6, 19, 18, 5, 33, 6, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a group of people standing on top of a snow covered slope', 'logprob': -4.210934162139893}, {'ids': [1, 4, 34, 6, 19, 5, 157, 8, 7, 74, 2, 0, 0, 0, 0, 0], 'caption': 'a group of people on skis in the snow', 'logprob': -4.96717643737793}, {'ids': [1, 4, 12, 18, 21, 13, 4, 156, 5, 4, 74, 132, 234, 2, 0, 0], 'caption': 'a man standing next to a child on a snow covered slope', 'logprob': -6.054995536804199}, {'ids': [1, 4, 12, 18, 21, 13, 16, 268, 5, 157, 2, 0, 0, 0, 0, 0], 'caption': 'a man standing next to two children on skis', 'logprob': -6.24704647064209}, {'ids': [1, 4, 34, 6, 19, 18, 5, 33, 6, 4, 74, 132, 269, 234, 2, 0], 'caption': 'a group of people standing on top of a snow covered ski slope', 'logprob': -6.3798041343688965}]}, 336431: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000336431.jpg', 'image_sha256': '4c17b31e2ecc89d08d97cf856fd8b725e361693ff649e5f0111b82df5c00434b', 'detected': {'chair': 0.9496, 'person': 0.9198, 'toothbrush': 0.8292}, 'selected': 1, 'beams': [{'ids': [1, 4, 50, 80, 14, 8, 4, 167, 27, 4, 662, 2, 0, 0, 0], 'caption': 'a young boy sitting in a chair holding a toothbrush', 'logprob': -6.721066474914551}, {'ids': [1, 4, 50, 80, 8, 4, 48, 178, 27, 4, 662, 2, 0, 0, 0], 'caption': 'a young boy in a red shirt holding a toothbrush', 'logprob': -6.982619285583496}, {'ids': [1, 4, 50, 80, 8, 4, 48, 178, 27, 35, 4, 662, 2, 0, 0], 'caption': 'a young boy in a red shirt holding up a toothbrush', 'logprob': -8.443425178527832}, {'ids': [1, 4, 50, 80, 8, 4, 48, 178, 27, 4, 499, 694, 2, 0, 0], 'caption': 'a young boy in a red shirt holding a plastic container', 'logprob': -9.069226264953613}, {'ids': [1, 4, 50, 80, 8, 4, 48, 178, 27, 35, 4, 499, 662, 2, 0], 'caption': 'a young boy in a red shirt holding up a plastic toothbrush', 'logprob': -10.289897918701172}]}, 366822: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000366822.jpg', 'image_sha256': '7fb7cffcf05cd409b74bf6faaeb2fac3a2db9ec4ed46fa332f43d02f716a744f', 'detected': {'person': 0.913, 'snowboard': 0.8957, 'backpack': 0.588}, 'selected': 0, 'beams': [{'ids': [1, 4, 34, 6, 19, 18, 21, 13, 135, 71, 8, 7, 74, 2, 0, 0], 'caption': 'a group of people standing next to each other in the snow', 'logprob': -4.072495460510254}, {'ids': [1, 4, 34, 6, 19, 26, 20, 18, 8, 7, 74, 2, 0, 0, 0, 0], 'caption': 'a group of people that are standing in the snow', 'logprob': -4.263339996337891}, {'ids': [1, 4, 34, 6, 19, 18, 21, 13, 135, 71, 2, 0, 0, 0, 0, 0], 'caption': 'a group of people standing next to each other', 'logprob': -4.424356460571289}, {'ids': [1, 4, 34, 6, 19, 18, 21, 13, 135, 71, 27, 157, 2, 0, 0, 0], 'caption': 'a group of people standing next to each other holding skis', 'logprob': -4.929179668426514}, {'ids': [1, 4, 34, 6, 19, 18, 21, 13, 135, 71, 27, 74, 573, 2, 0, 0], 'caption': 'a group of people standing next to each other holding snow boards', 'logprob': -4.960314750671387}]}, 528276: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000528276.jpg', 'image_sha256': 'e3477388ea64d498521cbd71ac751784c307c29752f662208b1794e83df7b4f8', 'detected': {'person': 0.8657, 'skateboard': 0.6064}, 'selected': 3, 'beams': [{'ids': [1, 4, 12, 47, 4, 91, 32, 4, 191, 2, 0, 0, 0, 0, 0], 'caption': 'a man riding a skateboard down a sidewalk', 'logprob': -3.593529224395752}, {'ids': [1, 4, 12, 47, 4, 91, 32, 4, 24, 2, 0, 0, 0, 0, 0], 'caption': 'a man riding a skateboard down a street', 'logprob': -5.066890716552734}, {'ids': [1, 4, 31, 47, 4, 248, 119, 8, 4, 138, 2, 0, 0, 0, 0], 'caption': 'a person riding a skate board in a park', 'logprob': -5.135272979736328}, {'ids': [1, 4, 12, 47, 4, 91, 21, 13, 4, 138, 2, 0, 0, 0, 0], 'caption': 'a man riding a skateboard next to a park', 'logprob': -5.61251974105835}, {'ids': [1, 4, 12, 47, 4, 91, 32, 4, 191, 21, 13, 4, 138, 2, 0], 'caption': 'a man riding a skateboard down a sidewalk next to a park', 'logprob': -7.265048503875732}]}, 557596: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000557596.jpg', 'image_sha256': 'bd666229b89cf45adb02d25ed48fbc475b28e4ce0813c1e21bda22b0a81d43d5', 'detected': {'person': 0.9384, 'tennis racket': 0.8663, 'chair': 0.6682}, 'selected': 0, 'beams': [{'ids': [1, 4, 23, 18, 5, 4, 38, 149, 27, 4, 352, 2, 0, 0, 0, 0], 'caption': 'a woman standing on a tennis court holding a racquet', 'logprob': -2.3026223182678223}, {'ids': [1, 4, 23, 27, 4, 38, 352, 5, 4, 38, 149, 2, 0, 0, 0, 0], 'caption': 'a woman holding a tennis racquet on a tennis court', 'logprob': -2.7982492446899414}, {'ids': [1, 4, 23, 297, 4, 38, 352, 5, 4, 38, 149, 2, 0, 0, 0, 0], 'caption': 'a woman swinging a tennis racquet on a tennis court', 'logprob': -3.7330708503723145}, {'ids': [1, 4, 23, 27, 4, 38, 352, 5, 33, 6, 4, 38, 149, 2, 0, 0], 'caption': 'a woman holding a tennis racquet on top of a tennis court', 'logprob': -4.237232685089111}, {'ids': [1, 4, 23, 18, 5, 33, 6, 4, 38, 149, 27, 4, 352, 2, 0, 0], 'caption': 'a woman standing on top of a tennis court holding a racquet', 'logprob': -4.495506286621094}]}}
CLIP={'id': 'openai/clip-vit-base-patch16', 'revision': '57c216476eefef5ab752ec549e440a49ae4ae5f3', 'files_sha256': {'merges.txt': '9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a', 'pytorch_model.bin': 'ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0', 'special_tokens_map.json': 'f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf', 'config.json': 'eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad', 'tokenizer.json': 'a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4', 'preprocessor_config.json': '910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd', 'tokenizer_config.json': '3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade', 'vocab.json': '3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69'}}
RECEIPT={'sha256': '80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24', 'updates': 1500, 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'parent_sha256': '56a90b2223b3bb64e433a342d048389dc3e80693fa574c2c1543e31e39a4377b'}
VOCAB_SHA='cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd'
PROMPT_PINS={'prompt_texts.json': '35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2', 'prompt_embeddings.npy': 'f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c'}


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    from kaggle_secrets import UserSecretsClient
    try:
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    except Exception:
        raise RuntimeError('Enable the named Kaggle API Secret with access to the source notebook') from None
    if not token:
        raise RuntimeError('The named Kaggle API Secret is empty')
    TOKENS.append(token)
    cli_env=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
        cli_env.pop(key,None)
    run([sys.executable,'-m','pip','install','-q','--target',CLI,'kaggle==2.2.5'],300,cli_env)
    cli_env.update(KAGGLE_API_TOKEN=token,PYTHONPATH=str(CLI)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1')
    downloaded=SOURCE/'scst_v94'
    downloaded.mkdir()
    run([sys.executable,'-u','-c','from kaggle.cli import main; main()',
         'kernels','output',SOURCE_VERSION,'-p',downloaded,
         '--file-pattern',r'(^|/)update1500/pilot_checkpoint[.]bin$'],900,cli_env)
    candidates=list(downloaded.rglob('pilot_checkpoint.bin'))
    if len(candidates)!=1:
        raise ValueError('Expected the SCST1500 checkpoint from exact v94')
    weights=candidates[0]
    with weights.open('rb') as stream:
        if hashlib.file_digest(stream,'sha256').hexdigest()!=RESUME_SHA:
            raise ValueError('Changed SCST1500 checkpoint')

    sys.path.insert(0,str(SOURCE))
    (SOURCE/'captioning').mkdir(exist_ok=True)
    (SOURCE/'captioning/__init__.py').touch()
    for path,name in MAPPING.items():
        if name in {'semantic_data.py','patch_model.py','patch_beam.py','patch_data.py','full_data.py','data.py','real_bank_data.py','cuda_runtime_guard.py'}:
            download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    for path,digest in TRAIN_PINS.items():
        name='bank_layers.py' if path.endswith('/model.py') else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+TRAIN_REVISION+'/'+path,digest,SOURCE/name)
    for path,digest in {'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/277851d0fbc4643ae7001bc7702dd0ce5c91d0f6/'+path,digest,target)

    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],300)
    sys.path.insert(0,str(DEPS))
    os.environ.update(TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')
    import numpy as np
    import torch
    import torch.nn.functional as F
    from PIL import Image
    from huggingface_hub import snapshot_download
    from transformers import CLIPModel, CLIPProcessor
    from captioning.multiscore_reranking import rerank_multiscore
    from csa_patch_model import CSAPatchCaptioner
    from patch_beam import beam_search, check_paths, text_from_ids
    from semantic_data import record_arrays
    from real_bank_data import sha

    if torch.__version__ not in ('2.11.0+cu128','2.11.0+cu130') or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU T4 with CUDA-enabled Torch 2.11.0')
    device=torch.device('cuda:0')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False

    mount=Path(CACHE_ROOT)
    with zipfile.ZipFile(mount/'cache_metadata_zip.bin') as z:
        receipt_bytes=z.read('part0/scst_cache_receipt.json')
        if hashlib.sha256(receipt_bytes).hexdigest()!='3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510':
            raise ValueError('Wrong SCST Part 0 receipt')
        cache=json.loads(receipt_bytes)['files_sha256']
        def member(name):
            raw=z.read('part0/'+name)
            if hashlib.sha256(raw).hexdigest()!=cache[name]:
                raise ValueError('Changed cache member '+name)
            return raw
        import io
        valid_ids=np.load(io.BytesIO(member('valid/ids.npy')),allow_pickle=False)
        raw_vocab=member('vocabulary.json')
        if hashlib.sha256(raw_vocab).hexdigest()!=VOCAB_SHA:
            raise ValueError('Wrong vocabulary for SCST1500')
        words=json.loads(raw_vocab)
    patchfile=mount/'part0_valid_patches.npy'
    if sha(patchfile)!=cache['valid/patches.npy']:
        raise ValueError('Changed VALID patches')
    patches=np.load(patchfile,mmap_mode='r',allow_pickle=False)
    index={int(cid):i for i,cid in enumerate(valid_ids)}
    if len(index)!=5000 or not SAVED.keys()<=index.keys():
        raise ValueError('Wrong VALID coverage')
    links,=PROMPT_TEST_ROOT.rglob('prompt_links.jsonl')
    if sha(links)!='e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0':
        raise ValueError('Wrong prompt175 links')
    texts_path=links.parent/'prompt_texts.json'
    vector_path=links.parent/'prompt_embeddings.npy'
    for path,digest in PROMPT_PINS.items():
        if sha(links.parent/path)!=digest:
            raise ValueError('Wrong prompt175 asset '+path)
    texts=json.loads(texts_path.read_text())
    vectors=np.load(vector_path,allow_pickle=False)
    with links.open(encoding='utf8') as stream:
        original={r['coco_id']:r for r in map(json.loads,stream) if r['group']=='val' and r['coco_id'] in SAVED}
    if set(original)!=set(SAVED):
        raise ValueError('Missing original VAL prompts')
    upt_path,=Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2').rglob('prompt_cache.json')
    if sha(upt_path)!='24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d':
        raise ValueError('Wrong UPT prompt source')
    upt=json.loads(upt_path.read_text(encoding='utf-8'))
    for name, text in {'prompt_adapter_data.py': '"""NumPy-only instance joins for the region-linked CSA prompt pilot."""\nfrom collections import Counter\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\n\nPROMPT_SHA = \'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d\'\nOBJECT_CAP, RELATION_CAP, IOU = 8, 3, .5\n\n\ndef label(value):\n    if not isinstance(value, str) or not value.strip():\n        raise ValueError(\'Missing object label\')\n    return \' \'.join(value.lower().replace(\'_\', \' \').split())\n\n\ndef box(value):\n    b = np.asarray(value, dtype=np.float64)\n    if b.shape != (4,) or not np.isfinite(b).all() or (b < 0).any() or (b > 1).any() or (b[2:] <= b[:2]).any():\n        raise ValueError(\'Expected normalized positive xyxy box\')\n    return b\n\n\ndef overlap(a, boxes):\n    a = box(a)\n    if not len(boxes):\n        return np.empty(0)\n    boxes = np.stack([box(b) for b in boxes])\n    intersection = np.maximum(0, np.minimum(a[2:], boxes[:, 2:]) - np.maximum(a[:2], boxes[:, :2])).prod(1)\n    return intersection / ((a[2:] - a[:2]).prod() + (boxes[:, 2:] - boxes[:, :2]).prod(1) - intersection)\n\n\ndef detector_regions(objects, width, height, saved_boxes, kinds):\n    """Reconstruct the existing cache\'s confidence-sort/clip/cap before label joins."""\n    if width <= 0 or height <= 0 or len(kinds) != len(saved_boxes):\n        raise ValueError(\'Invalid ROI dimensions\')\n    ordered = sorted(enumerate(objects), key=lambda x: (-float(x[1][\'conf\']), x[0]))[:150]\n    if not ordered:\n        if kinds != [\'full_frame_fallback\'] or not np.allclose(saved_boxes, [[0, 0, width, height]], rtol=0, atol=1e-4):\n            raise ValueError(\'Changed fallback ROI\')\n        return []\n    if kinds != [\'predicted_roi\'] * len(ordered):\n        raise ValueError(\'Detector and ROI counts/kinds disagree\')\n    result = []\n    for i, (_, obj) in enumerate(ordered):\n        score = float(obj[\'conf\'])\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid detector confidence\')\n        b = np.clip(np.asarray(obj[\'bbox\'], np.float32), [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if b.shape != (4,) or not np.isfinite(b).all() or not np.allclose(b, saved_boxes[i], atol=1e-4, rtol=0):\n            raise ValueError(\'Detector order/box differs from saved ROI\')\n        result.append(dict(roi=i, label=label(obj[\'label\']), bbox=box(b / [width, height, width, height]).tolist()))\n    return result\n\n\ndef geometry(subject, obj):\n    s, o = box(subject), box(obj)\n    sw, sh = s[2:] - s[:2]; ow, oh = o[2:] - o[:2]\n    sc, oc = (s[:2] + s[2:]) / 2, (o[:2] + o[2:]) / 2\n    intersection = np.maximum(0, np.minimum(s[2:], o[2:]) - np.maximum(s[:2], o[:2])).prod()\n    return [float(v) for v in ((oc[0]-sc[0])/sw, (oc[1]-sc[1])/sh, np.log(ow/sw), np.log(oh/sh),\n            intersection/(sw*sh+ow*oh-intersection), intersection/(sw*sh), intersection/(ow*oh), np.linalg.norm(oc-sc))]\n\n\ndef prompt_text(name):\n    # Use the established UPT article handling, including plural skis/scissors.\n    from captioning.upt_prompts import noun\n    return noun(name)\n\n\ndef join_entry(entry, regions):\n    """Only selected objects/relations are consumed; previews/refs/GT are ignored."""\n    if entry[\'mode\'] not in (\'object_relation\', \'object_only\', \'no_objects\'):\n        raise ValueError(\'Unknown prompt mode\')\n    objects = entry[\'objects\']; relations = entry[\'relations\']\n    if not isinstance(objects, list) or not isinstance(relations, list) or len(relations) > RELATION_CAP:\n        raise ValueError(\'Malformed or unapproved prompt caps\')\n    by_index = {}; instances = set(); mapped = {}; stats = Counter(objects=len(objects), relations=len(relations))\n    for obj in objects:\n        index = obj[\'index\']; instance = obj[\'instance_id\']; name = label(obj[\'label\']); b = box(obj[\'bbox\'])\n        score = float(obj[\'confidence\'])\n        if type(index) != int or index in by_index or not isinstance(instance, str) or not instance or instance in instances:\n            raise ValueError(\'Duplicate/invalid object identity\')\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid UPT object score\')\n        by_index[index] = obj; instances.add(instance)\n        candidates = [r for r in regions if r[\'label\'] == name]\n        values = overlap(b, [r[\'bbox\'] for r in candidates])\n        if not len(values) or values.max() < IOU:\n            stats[\'unmatched_objects\'] += 1; continue\n        best = np.flatnonzero(np.isclose(values, values.max(), rtol=0, atol=1e-8))\n        if len(best) != 1:\n            stats[\'ambiguous_objects\'] += 1; continue\n        j = int(best[0]); r = candidates[j]\n        mapped[index] = dict(instance_id=instance, upt_index=index, roi=r[\'roi\'], label=name, bbox=b.tolist(),\n            roi_bbox=r[\'bbox\'], IoU=float(values[j]), confidence=score, text=prompt_text(name))\n    collisions = Counter(r[\'roi\'] for r in mapped.values())\n    for index in list(mapped):\n        if collisions[mapped[index][\'roi\']] > 1:\n            del mapped[index]; stats[\'colliding_objects\'] += 1\n    endpoints = []; valid_relations = []; seen_pairs = set()\n    for r in relations:\n        s, o = r[\'subject_index\'], r[\'object_index\']\n        if type(s) != int or type(o) != int or s == o or s not in by_index or o not in by_index:\n            raise ValueError(\'Broken relation endpoints\')\n        if r[\'subject_instance\'] != by_index[s][\'instance_id\'] or r[\'object_instance\'] != by_index[o][\'instance_id\']:\n            raise ValueError(\'Relation instance IDs differ\')\n        if label(r[\'subject\']) != \'person\' or label(by_index[s][\'label\']) != \'person\' or label(r[\'object\']) != label(by_index[o][\'label\']):\n            raise ValueError(\'Wrong subject/object role labels\')\n        score, threshold = float(r[\'score\']), float(r[\'threshold\'])\n        if not np.isfinite([score, threshold]).all() or not 0 <= score <= 1 or not 0 <= threshold <= 1.01 or score < threshold:\n            raise ValueError(\'Rejected/nonfinite relation in selected list\')\n        from captioning.upt_prompts import relation_text\n        expected = relation_text(r[\'predicate\'], label(r[\'object\']))\n        if r[\'text\'] != expected:\n            raise ValueError(\'Relation text differs from locked UPT template\')\n        key = (s, r[\'predicate\'], o)\n        if key in seen_pairs:\n            raise ValueError(\'Duplicate relation\')\n        seen_pairs.add(key)\n        if s not in mapped or o not in mapped:\n            stats[\'unmatched_relations\'] += 1; continue\n        endpoints.extend((s, o))\n        valid_relations.append(dict(subject_instance=r[\'subject_instance\'], object_instance=r[\'object_instance\'],\n            subject_roi=mapped[s][\'roi\'], object_roi=mapped[o][\'roi\'], text=expected, predicate=r[\'predicate\'],\n            score=score, threshold=threshold, geometry=geometry(by_index[s][\'bbox\'], by_index[o][\'bbox\'])))\n    order = list(dict.fromkeys(endpoints))\n    order += [i for i in sorted(mapped, key=lambda i: (-mapped[i][\'confidence\'], i)) if i not in order]\n    selected = [mapped[i] for i in order[:OBJECT_CAP]]\n    stats.update(mapped_objects=len(mapped), kept_objects=len(selected), mapped_relations=len(valid_relations),\n                 capped_objects=max(0, len(mapped)-OBJECT_CAP))\n    return dict(source_mode=entry[\'mode\'], objects=selected, relations=valid_relations, counts=dict(stats))\n\n\ndef cohort(ids):\n    ids = [int(i) for i in ids]\n    if len(ids) != len(set(ids)) or len(ids) < 10000:\n        raise ValueError(\'Need unique Karpathy TRAIN IDs\')\n    return sorted(ids, key=lambda i: hashlib.sha256(f\'42:{i}\'.encode()).hexdigest())[:10000]\n\n\ndef inference_record(meta, linked):\n    """Do not copy captions, sequences or any supervision into prompt payloads."""\n    return dict(coco_id=int(meta[\'coco_id\']), filename=meta[\'filename\'], group=meta[\'group\'],\n                roi_count=len(meta[\'region_kinds\']), **linked)\n\n\ndef tensor_batch(records, embeddings, texts):\n    """Data-only packing shared by the future worker and neural smoke tests."""\n    lookup = {s: i for i, s in enumerate(texts)}\n    if len(lookup) != len(texts) or embeddings.shape != (len(texts), 512) or not np.isfinite(embeddings).all():\n        raise ValueError(\'Invalid prompt embedding cache\')\n    n = len(records)\n    out = dict(object_text=np.zeros((n,8,512),np.float32), object_roi=np.zeros((n,8),np.int64),\n        object_box=np.zeros((n,8,4),np.float32), object_valid=np.zeros((n,8),bool),\n        relation_text=np.zeros((n,3,512),np.float32), relation_pair=np.zeros((n,3,2),np.int64),\n        relation_geometry=np.zeros((n,3,8),np.float32), relation_valid=np.zeros((n,3),bool))\n    for i, row in enumerate(records):\n        if len(row[\'objects\']) > 8 or len(row[\'relations\']) > 3:\n            raise ValueError(\'Prompt slot overflow\')\n        for j, obj in enumerate(row[\'objects\']):\n            if not 0 <= obj[\'roi\'] < row[\'roi_count\']:\n                raise ValueError(\'Object ROI out of bounds\')\n            out[\'object_text\'][i,j]=embeddings[lookup[obj[\'text\']]];out[\'object_roi\'][i,j]=obj[\'roi\']\n            out[\'object_box\'][i,j]=box(obj[\'bbox\']);out[\'object_valid\'][i,j]=True\n        for j, rel in enumerate(row[\'relations\']):\n            pair=[rel[\'subject_roi\'],rel[\'object_roi\']]\n            if pair[0] == pair[1] or any(not 0 <= k < row[\'roi_count\'] for k in pair):\n                raise ValueError(\'Invalid relation pair\')\n            g=np.asarray(rel[\'geometry\'],np.float32)\n            if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid pair geometry\')\n            out[\'relation_text\'][i,j]=embeddings[lookup[rel[\'text\']]];out[\'relation_pair\'][i,j]=pair\n            out[\'relation_geometry\'][i,j]=g;out[\'relation_valid\'][i,j]=True\n    return out\n', 'captioning/upt_prompts.py': '"""UPT predictions -> instance-linked text; no annotations or reference captions."""\nimport math\n\nfrom .selected_relation_prompts import entity_id, phrase\nfrom .upt_action_text import ACTION_TEXT, action_predicate\n\nMETHOD = \'upt_object_relation_prompt_v1\'\nPREDICATES = (\'holding\', \'riding\')\n\n\ndef noun(label):\n    return label if label in (\'skis\', \'scissors\') else phrase(label)\n\n\ndef relation_text(predicate, obj):\n    verb = {\'holding\': \'hold\', \'riding\': \'ride\'}.get(predicate, predicate)\n    if verb not in ACTION_TEXT:\n        raise ValueError(\'Unsupported prompt predicate.\')\n    # Do not infer surfing/skiing from object class alone. Keep UPT\'s broad ride.\n    if predicate == \'riding\' and obj == \'skis\':\n        return \'a person on skis\'\n    if verb in (\'jump\', \'run\'):\n        # These labels do not specify the object\'s grammatical role; do not invent one.\n        return f\'a person and {noun(obj)} (interaction: {ACTION_TEXT[verb]})\'\n    return f\'a person {ACTION_TEXT[verb]} {noun(obj)}\'\n\n\ndef from_audit(row):\n    """Allowlist inference fields; deliberately discard every GT/matched ID/label."""\n    detections = [{k: d[k] for k in (\'label\', \'confidence\', \'box\')} for d in row[\'detection_details\']]\n    claims = [{k: c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\')} for c in row[\'candidates\']]\n    return detections, claims\n\n\ndef from_cache(row, vocabulary, all_actions=False):\n    h, w = row[\'resized_size_hw\']\n    if h <= 0 or w <= 0 or len(row[\'boxes_resized_xyxy\']) != len(row[\'labels\']) or len(row[\'labels\']) != len(row[\'object_scores\']):\n        raise ValueError(\'Invalid cache detection dimensions.\')\n    detections = []\n    for box, label, score in zip(row[\'boxes_resized_xyxy\'], row[\'labels\'], row[\'object_scores\']):\n        if not 0 <= label < len(vocabulary[\'objects\']):\n            raise ValueError(\'Invalid cache object label.\')\n        detections.append({\'label\': vocabulary[\'objects\'][label].replace(\'_\', \' \'), \'confidence\': score,\n                           \'box\': [max(0., min(1., x / scale)) for x, scale in zip(box, (w, h, w, h))]})\n    if len(row[\'pairs\']) != len(row[\'native_scores\']):\n        raise ValueError(\'Invalid native score dimensions.\')\n    claims = []\n    for (s, o), scores in zip(row[\'pairs\'], row[\'native_scores\']):\n        if not 0 <= s < len(detections) or not 0 <= o < len(detections):\n            raise ValueError(\'Invalid cache pair indices.\')\n        verbs = vocabulary[\'verbs_by_object\'][row[\'labels\'][o]]\n        if len(scores) != len(verbs):\n            raise ValueError(\'Sparse native score row differs from ontology.\')\n        for verb, score in zip(verbs, scores):\n            name = vocabulary[\'verbs\'][verb]\n            if name == \'no_interaction\':\n                continue\n            if all_actions and name not in ACTION_TEXT:\n                raise ValueError(f\'Unknown UPT action: {name}\')\n            if all_actions or name in (\'hold\', \'ride\'):\n                claims.append({\'subject\': s, \'object\': o, \'predicate\': action_predicate(name),\n                               \'score\': score})\n    return detections, claims\n\n\ndef build_prompt(filename, detections, claims, policy, max_relations=3, object_threshold=.5, max_object_names=8):\n    if max_relations < 1 or max_object_names < 1 or not math.isfinite(object_threshold) or not 0 <= object_threshold <= 1:\n        raise ValueError(\'Invalid prompt options.\')\n    thresholds = {p: value[\'threshold\'] for p, value in policy[\'predicates\'].items()}\n    if not thresholds or any(p not in {action_predicate(v) for v in ACTION_TEXT} for p in thresholds):\n        raise ValueError(\'Unknown or empty policy predicates.\')\n    if any(not math.isfinite(t) or not 0 <= t <= 1.01 for t in thresholds.values()):\n        raise ValueError(\'Invalid locked policy threshold.\')\n    objects = []\n    for i, d in enumerate(detections):\n        label, confidence, box = d[\'label\'], d[\'confidence\'], d[\'box\']\n        if not isinstance(label, str) or not label.strip() or label != label.strip():\n            raise ValueError(\'Invalid object name.\')\n        if not math.isfinite(confidence) or not 0 <= confidence <= 1:\n            raise ValueError(\'Invalid object confidence.\')\n        if len(box) != 4 or not all(math.isfinite(v) for v in box) or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1:\n            raise ValueError(\'Invalid normalized predicted box.\')\n        objects.append({\'index\': i, \'instance_id\': entity_id(filename, label, box),\n                        \'label\': label, \'confidence\': confidence, \'bbox\': box})\n    unique = {}\n    for c in claims:\n        s, o, p, score = (c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\'))\n        if p not in thresholds or s == o or not 0 <= s < len(objects) or not 0 <= o < len(objects):\n            raise ValueError(\'Invalid relation pair/predicate.\')\n        if objects[s][\'label\'] != \'person\' or not math.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid relation actor/score.\')\n        key = (s, p, o)\n        if key not in unique or score > unique[key][\'score\']:\n            unique[key] = {\'subject_index\': s, \'object_index\': o, \'subject_instance\': objects[s][\'instance_id\'],\n                           \'object_instance\': objects[o][\'instance_id\'], \'subject\': \'person\', \'object\': objects[o][\'label\'],\n                           \'predicate\': p, \'score\': score, \'threshold\': thresholds[p],\n                           \'text\': relation_text(p, objects[o][\'label\'])}\n    # Margin is only a fixed cap ordering; not calibrated cross-predicate confidence.\n    accepted = sorted((c for c in unique.values() if c[\'score\'] >= c[\'threshold\']),\n                      key=lambda c: (-(c[\'score\'] - c[\'threshold\']), c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))\n    selected = accepted[:max_relations]\n    endpoints = {c[k] for c in selected for k in (\'subject_index\', \'object_index\')}\n    kept = [d for d in objects if d[\'confidence\'] >= object_threshold or d[\'index\'] in endpoints]\n    names = list(dict.fromkeys(d[\'label\'] for d in sorted(kept, key=lambda d: (-d[\'confidence\'], d[\'index\']))))[:max_object_names]\n    object_prompt = \'a photo containing \' + \', \'.join(noun(n) for n in names) + \'.\' if names else \'\'\n    texts = list(dict.fromkeys(c[\'text\'] for c in selected))\n    relation_prompt = \'a photo showing \' + \'; \'.join(texts) + \'.\' if texts else \'\'\n    previews = sorted(unique.values(), key=lambda c: (-c[\'score\'], c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))[:3]\n    return {\'objects\': kept, \'relations\': selected, \'object_prompt\': object_prompt,\n            \'relation_prompt\': relation_prompt, \'prompt\': relation_prompt or object_prompt or \'a photo.\',\n            \'mode\': \'object_relation\' if selected else \'object_only\' if names else \'no_objects\',\n            \'above_threshold_claims\': len(accepted), \'capped_claims\': len(accepted) - len(selected),\n            \'candidate_preview\': [{**c, \'passes_locked_threshold\': c[\'score\'] >= c[\'threshold\']} for c in previews]}\n', 'captioning/upt_action_text.py': '"""Explicit text for all 116 positive HICO action labels; no new action inference."""\n\nACTION_TEXT = dict(line.split(\'=\', 1) for line in \'\'\'adjust=adjusting\nassemble=assembling\nblock=blocking\nblow=blowing toward\nboard=boarding\nbreak=breaking\nbrush_with=brushing with\nbuy=buying\ncarry=carrying\ncatch=catching\nchase=chasing\ncheck=checking\nclean=cleaning\ncontrol=controlling\ncook=cooking\ncut=cutting\ncut_with=cutting with\ndirect=directing\ndrag=dragging\ndribble=dribbling\ndrink_with=drinking from\ndrive=driving\ndry=drying\neat=eating\neat_at=eating at\nexit=exiting\nfeed=feeding\nfill=filling\nflip=flipping\nflush=flushing\nfly=flying\ngreet=greeting\ngrind=grinding on\ngroom=grooming\nherd=herding\nhit=hitting\nhold=holding\nhop_on=hopping on\nhose=hosing\nhug=hugging\nhunt=hunting\ninspect=inspecting\ninstall=installing\njump=jumping\nkick=kicking\nkiss=kissing\nlasso=lassoing\nlaunch=launching\nlick=licking\nlie_on=lying on\nlift=lifting\nlight=lighting\nload=loading\nlose=losing\nmake=making\nmilk=milking\nmove=moving\nopen=opening\noperate=operating\npack=packing\npaint=painting\npark=parking\npay=paying at\npeel=peeling\npet=petting\npick=picking\npick_up=picking up\npoint=pointing\npour=pouring from\npull=pulling\npush=pushing\nrace=racing\nread=reading\nrelease=releasing\nrepair=repairing\nride=riding\nrow=rowing\nrun=running\nsail=sailing\nscratch=scratching\nserve=serving\nset=setting\nshear=shearing\nsign=signing\nsip=sipping from\nsit_at=sitting at\nsit_on=sitting on\nslide=sliding on\nsmell=smelling\nspin=spinning\nsqueeze=squeezing\nstab=stabbing\nstand_on=standing on\nstand_under=standing under\nstick=sticking\nstir=stirring\nstop_at=stopping at\nstraddle=straddling\nswing=swinging\ntag=tagging\ntalk_on=talking on\nteach=teaching\ntext_on=texting on\nthrow=throwing\ntie=tying\ntoast=toasting with\ntrain=training\nturn=turning\ntype_on=typing on\nwalk=walking\nwash=washing\nwatch=watching\nwave=waving at\nwear=wearing\nwield=wielding\nzip=zipping\'\'\'.splitlines())\n\nassert len(ACTION_TEXT) == 116 and \'no_interaction\' not in ACTION_TEXT\n\n\ndef action_predicate(verb):\n    return {\'hold\': \'holding\', \'ride\': \'riding\'}.get(verb, verb)\n\n\ndef uniform_policy(threshold=.5):\n    import math\n    if not math.isfinite(threshold) or not 0 < threshold <= 1:\n        raise ValueError(\'Exploratory threshold must be in (0,1].\')\n    return {\'mode\': \'all_actions_exploratory\', \'calibrated\': False,\n            \'predicates\': {action_predicate(v): {\'threshold\': threshold} for v in ACTION_TEXT}}\n', 'captioning/selected_relation_prompts.py': '"""Select scored detector claims without consulting annotation IDs or GT labels."""\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict\nfrom captioning.supervised_relations import RELATIONS\n\n\ndef entity_id(filename, label, box):\n    payload = json.dumps([filename, label, box], separators=(\',\', \':\'))\n    return \'det_\' + hashlib.sha256(payload.encode()).hexdigest()[:12]\n\n\ndef phrase(label):\n    article = \'an\' if label[:1].lower() in \'aeiou\' else \'a\'\n    return f\'{article} {label}\'\n\n\ndef build_selected_prompts(scored_pairs, thresholds, max_relations=3, object_inventory=None):\n    if len(thresholds) != len(RELATIONS) or max_relations < 1:\n        raise ValueError(\'Invalid thresholds/relation limit.\')\n    if not all(math.isfinite(x) for x in thresholds):\n        raise ValueError(\'Thresholds must be finite.\')\n    groups = defaultdict(list)\n    for filename in object_inventory or {}:\n        groups[filename] = []\n    for row in scored_pairs:\n        if len(row[\'scores\']) != len(RELATIONS) or not all(math.isfinite(v) for v in row[\'scores\']):\n            raise ValueError(\'Invalid scores.\')\n        for role in (\'subject\', \'object\'):\n            box = row[role+\'_box\']\n            if (len(box) != 4 or not all(math.isfinite(v) for v in box)\n                    or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1):\n                raise ValueError(\'Expected normalized nonempty detector boxes.\')\n        groups[row[\'filename\']].append(row)\n    result = {}\n    for filename, rows in sorted(groups.items()):\n        objects, candidates = {}, {}\n        for entity in (object_inventory or {}).get(filename, []):\n            identity = entity_id(filename, entity[\'label\'], entity[\'bbox\'])\n            objects[identity] = {**entity, \'instance_id\': identity,\n                                 \'prompt\': \'a photo of \'+phrase(entity[\'label\'])}\n        for row in rows:\n            ids = []\n            for role in (\'subject\', \'object\'):\n                box, label = row[role+\'_box\'], row[role]\n                identity = entity_id(filename, label, box)\n                ids.append(identity)\n                objects[identity] = {\'instance_id\': identity, \'label\': label, \'bbox\': box,\n                    \'confidence\': row.get(role+\'_detector_confidence\'), \'prompt\': \'a photo of \'+phrase(label)}\n            for c, relation in enumerate(RELATIONS):\n                score = row[\'scores\'][c]\n                if score < thresholds[c] or ids[0] == ids[1]:\n                    continue\n                key = (ids[0], relation, ids[1])\n                claim = {\'subject_instance\': ids[0], \'subject\': row[\'subject\'],\n                         \'object_instance\': ids[1], \'object\': row[\'object\'], \'relation\': relation,\n                         \'score\': score, \'threshold\': thresholds[c], \'score_margin\': score-thresholds[c],\n                         \'prompt\': f\'a photo of {phrase(row["subject"])} {relation.replace("_", " ")} {phrase(row["object"])}\'}\n                if key not in candidates or score > candidates[key][\'score\']:\n                    candidates[key] = claim\n        # Margin is a deterministic cap priority, NOT calibrated cross-class confidence.\n        ordered = sorted(candidates.values(), key=lambda c: (-c[\'score_margin\'], c[\'subject_instance\'],\n                                                            c[\'relation\'], c[\'object_instance\']))\n        selected = ordered[:max_relations]\n        labels = sorted({entity[\'label\'] for entity in objects.values()})\n        object_text = \'a photo of \'+\', \'.join(phrase(label) for label in labels) if labels else \'\'\n        relation_text = (\'a photo showing \'+\', \'.join(\n            f\'{phrase(c["subject"])} {c["relation"].replace("_", " ")} {phrase(c["object"])}\' for c in selected))\n        result[filename] = {\'objects\': list(objects.values()), \'relations\': selected,\n                            \'object_prompt\': object_text,\n                            \'relation_prompt\': relation_text if selected else \'\',\n                            \'prompt\': relation_text if selected else object_text,\n                            \'mode\': \'object_relation\' if selected else \'object_only\' if objects else \'no_objects\',\n                            \'above_threshold_claims\': len(ordered), \'capped_claims\': len(ordered)-len(selected)}\n    return result\n', 'captioning/supervised_relations.py': '"""V-COCO pair supervision. Missing links are UNKNOWN, never invented negatives."""\nimport math\nimport random\nfrom collections import Counter, defaultdict\n\nRELATION_SPECS = (\n    (\'hold\', \'obj\', \'holding\'), (\'ride\', \'instr\', \'riding\'),\n    (\'sit\', \'instr\', \'sitting_on\'), (\'carry\', \'obj\', \'carrying\'),\n    (\'eat\', \'obj\', \'eating\'), (\'talk_on_phone\', \'instr\', \'talking_on\'),\n)\nRELATIONS = [spec[2] for spec in RELATION_SPECS]\n\n\ndef decode_action(action):\n    """Official serialization stores all agents, then each role column."""\n    n = len(action[\'ann_id\'])\n    roles = action[\'role_name\']\n    flat = action[\'role_object_id\']\n    if n == 0 or len(flat) != n * len(roles):\n        raise ValueError(\'Invalid V-COCO role array.\')\n    if len(action[\'image_id\']) != n or len(action[\'label\']) != n:\n        raise ValueError(\'V-COCO column lengths differ.\')\n    if roles[0] != \'agent\' or flat[:n] != action[\'ann_id\']:\n        raise ValueError(\'Agent column does not match annotation IDs.\')\n    rows = {}\n    for i, sid in enumerate(action[\'ann_id\']):\n        if sid in rows or action[\'label\'][i] not in (-1, 0, 1):\n            raise ValueError(\'Duplicate actor or invalid action label.\')\n        rows[sid] = {\'image_id\': action[\'image_id\'][i], \'label\': action[\'label\'][i],\n                     \'roles\': {role: flat[j * n + i] for j, role in enumerate(roles)}}\n    return rows\n\n\ndef normalized_box(annotation, image):\n    x, y, w, h = annotation[\'bbox\']\n    width, height = image[\'width\'], image[\'height\']\n    if not all(math.isfinite(v) for v in (x, y, w, h, width, height)):\n        raise ValueError(\'Non-finite box.\')\n    if min(w, h, width, height) <= 0:\n        raise ValueError(\'Empty box/image.\')\n    box = [max(0, x / width), max(0, y / height),\n           min(1, (x + w) / width), min(1, (y + h) / height)]\n    if not (box[0] < box[2] and box[1] < box[3]):\n        raise ValueError(\'Box outside image.\')\n    return box\n\n\ndef build_pairs(actions, instances, caption_images, max_objects=12):\n    if max_objects < 1:\n        raise ValueError(\'max_objects must be positive.\')\n    by_action = {a[\'action_name\']: decode_action(a) for a in actions}\n    if any(name not in by_action for name, _, _ in RELATION_SPECS):\n        raise ValueError(\'Required V-COCO action missing.\')\n    images = {x[\'id\']: x for x in instances[\'images\']}\n    anns = {x[\'id\']: x for x in instances[\'annotations\']}\n    categories = {x[\'id\']: x[\'name\'] for x in instances[\'categories\']}\n    caption = {x[\'cocoid\']: x for x in caption_images}\n    if len(caption) != len(caption_images):\n        raise ValueError(\'Duplicate caption image ID.\')\n    by_image = defaultdict(list)\n    boxes = {}\n    for aid, ann in anns.items():\n        if ann.get(\'iscrowd\', 0):\n            continue\n        boxes[aid] = normalized_box(ann, images[ann[\'image_id\']])\n        by_image[ann[\'image_id\']].append(aid)\n    subjects = sorted(set(s for a in by_action.values() for s in a))\n    rows, stats = [], Counter()\n    for sid in subjects:\n        if sid not in anns:\n            raise ValueError(f\'Missing actor annotation {sid}.\')\n        ann = anns[sid]\n        iid = ann[\'image_id\']\n        if iid not in caption:\n            raise ValueError(f\'Image {iid} missing from caption split metadata.\')\n        if caption[iid][\'split\'] not in {\'train\', \'restval\'}:\n            stats[\'excluded_caption_heldout_subjects\'] += 1\n            continue\n        if sid not in boxes:\n            stats[\'excluded_crowd_subjects\'] += 1\n            continue\n        if categories[ann[\'category_id\']] != \'person\':\n            raise ValueError(\'V-COCO agent must be a person.\')\n        positives = set()\n        for name, role, _ in RELATION_SPECS:\n            item = by_action[name].get(sid)\n            if item and item[\'image_id\'] != iid:\n                raise ValueError(\'Actor/image ID mismatch.\')\n            if item and item[\'label\'] == 1:\n                oid = item[\'roles\'].get(role, 0)\n                if oid:\n                    if oid not in anns or anns[oid][\'image_id\'] != iid or oid == sid:\n                        raise ValueError(\'Invalid positive role object.\')\n                    if oid in boxes:\n                        positives.add(oid)\n                    else:\n                        stats[\'unusable_positive_box\'] += 1\n                else:\n                    stats[\'positive_action_without_visible_role\'] += 1\n        # Keep every positive; fill with deterministic same-image candidates.\n        candidates = sorted(set(by_image[iid]) - {sid} - positives)\n        candidates = sorted(positives) + candidates[:max(0, max_objects - len(positives))]\n        for oid in candidates:\n            labels = []\n            for name, role, _ in RELATION_SPECS:\n                item = by_action[name].get(sid)\n                label = -1\n                if item and item[\'label\'] == 0:\n                    label = 0  # Annotated absence of this actor action implies absence for any B.\n                elif item and item[\'label\'] == 1 and item[\'roles\'].get(role, 0) == oid:\n                    label = 1\n                # label=1 with another B, missing role, or label=-1 remains UNKNOWN.\n                labels.append(label)\n            if all(v == -1 for v in labels):\n                continue\n            image = caption[iid]\n            rows.append({\'image_id\': iid, \'filename\': image[\'filename\'],\n                         \'filepath\': image.get(\'filepath\', \'\'), \'subject_id\': sid,\n                         \'object_id\': oid, \'subject\': \'person\',\n                         \'object\': categories[anns[oid][\'category_id\']],\n                         \'subject_box\': boxes[sid], \'object_box\': boxes[oid],\n                         \'labels\': labels, \'caption_split\': image[\'split\']})\n    return rows, dict(stats)\n\n\ndef diagnostic_subset(rows, limit, seed=42):\n    if limit < 1:\n        raise ValueError(\'Pair limit must be positive.\')\n    rng = random.Random(seed)\n    pos = [r for r in rows if 1 in r[\'labels\']]\n    neg = [r for r in rows if 1 not in r[\'labels\']]\n    rng.shuffle(pos)\n    rng.shuffle(neg)\n    by_target = defaultdict(list)\n    for row in rows:\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(row)\n    # Enrich a bounded portion with confirmed actor-counterpart anchors; this is\n    # explicitly a development diagnostic, not a prevalence-preserving benchmark.\n    actor_anchors = [row for row in pos if any(\n        other[\'subject_id\'] != row[\'subject_id\']\n        and any(a == 1 and b == 0 for a, b in zip(row[\'labels\'], other[\'labels\']))\n        for other in by_target[(row[\'image_id\'], row[\'object_id\'])])]\n    chosen = actor_anchors[:limit // 8]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    chosen += [r for r in pos if (r[\'subject_id\'], r[\'object_id\']) not in used][\n        :max(0, min(len(pos), limit // 2) - len(chosen))]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    for anchor in list(chosen):\n        candidates = by_target[(anchor[\'image_id\'], anchor[\'object_id\'])]\n        for other in candidates:\n            key = (other[\'subject_id\'], other[\'object_id\'])\n            if (key not in used and other[\'subject_id\'] != anchor[\'subject_id\']\n                    and any(a == 1 and b == 0 for a, b in zip(anchor[\'labels\'], other[\'labels\']))):\n                chosen.append(other)\n                used.add(key)\n                break\n        if len(chosen) >= limit:\n            break\n    # Fill remaining slots with all-zero/unknown pairs before more positives.\n    # Positive-first fill previously skewed the intended diagnostic composition.\n    remaining = [r for r in neg + pos if (r[\'subject_id\'], r[\'object_id\']) not in used]\n    chosen += remaining[:max(0, limit - len(chosen))]\n    return sorted(chosen, key=lambda r: (r[\'image_id\'], r[\'subject_id\'], r[\'object_id\']))\n\n\ndef contrast_edges(rows, max_per_positive=4):\n    """Only annotated-zero claims can be negative; no role swaps inferred false."""\n    by_target = defaultdict(list)\n    for i, row in enumerate(rows):\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(i)\n    edges = []\n    for i, row in enumerate(rows):\n        for r, label in enumerate(row[\'labels\']):\n            if label != 1:\n                continue\n            wrong_pred = [(i, c, \'predicate\') for c, v in enumerate(row[\'labels\']) if v == 0]\n            wrong_actor = [(j, r, \'actor\') for j in by_target[(row[\'image_id\'], row[\'object_id\'])]\n                           if rows[j][\'subject_id\'] != row[\'subject_id\'] and rows[j][\'labels\'][r] == 0]\n            # Alternate types so a predicate-heavy anchor does not exclude all actor edges.\n            negatives = []\n            while wrong_pred or wrong_actor:\n                if wrong_actor:\n                    negatives.append(wrong_actor.pop(0))\n                if wrong_pred:\n                    negatives.append(wrong_pred.pop(0))\n            edges += [(i, r, j, c, kind) for j, c, kind in negatives[:max_per_positive]]\n    return edges\n\n\ndef pair_statistics(rows):\n    return {\'pairs\': len(rows), \'images\': len({r[\'image_id\'] for r in rows}),\n            \'labels\': {name: dict(Counter(str(r[\'labels\'][c]) for r in rows))\n                       for c, name in enumerate(RELATIONS)},\n            \'contrast_edges\': dict(Counter(e[-1] for e in contrast_edges(rows)))}\n\n\ndef select_unseen_pairs(rows, seen, count=500, seed=20261013):\n    eligible = [r for r in rows if r[\'image_id\'] not in seen]\n    selected = diagnostic_subset(eligible, count, seed=seed)\n    if len(selected) != count:\n        raise ValueError(\'Insufficient new pairs; do not silently change protocol.\')\n    if {r[\'image_id\'] for r in selected} & set(seen):\n        raise ValueError(\'Seen image selected.\')\n    return selected, eligible\n'}.items():
        path=SOURCE/name
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text(text,encoding='utf-8')
    from copy import deepcopy
    edited=deepcopy(original)
    applied=[]
    blocked=[]
    for p in PROPOSALS:
        row=edited[p['coco_id']]
        strict=original[p['coco_id']]
        entry=upt[p['filename']]
        obj={o['instance_id']:o for o in entry['objects']}
        by_index={o['index']:o for o in entry['objects']}
        source=[r for r in entry['relations'] if r['object_instance']==p['object_instance'] and r['predicate']==p['predicate']
                and any(o['instance_id']==r['subject_instance'] and o['roi_bbox']==p['subject_roi'] for o in strict['objects'])]
        if len(source)!=1:
            raise ValueError('Proposal does not resolve to one strict subject and UPT relation')
        r=source[0]
        kept={o['instance_id']:o for o in row['objects']}
        oid=r['object_instance']
        if len(row['relations'])>=3 or (oid not in kept and len(row['objects'])>=8):
            blocked.append(dict(case_id=p['case_id'],reason='slot_capacity'))
            continue
        o=obj[oid]
        if oid not in kept:
            if any(x['roi']==p['object_roi'] for x in kept.values()):
                raise ValueError('Proposed ROI is already occupied')
            from captioning.upt_prompts import noun
            row['objects'].append(dict(instance_id=oid,roi=p['object_roi'],bbox=o['bbox'],
                                       roi_bbox=p['yolo_object'],text=noun(o['label'])))
        from prompt_adapter_data import geometry
        row['relations'].append(dict(subject_instance=r['subject_instance'],object_instance=oid,text=r['text'],
                                     geometry=geometry(by_index[r['subject_index']]['bbox'],o['bbox'])))
        applied.append(p['case_id'])
    del upt
    if len(applied)+len(blocked)!=29:
        raise ValueError('Incomplete proposal accounting')

    root=Path(snapshot_download(CLIP['id'],revision=CLIP['revision'],allow_patterns=list(CLIP['files_sha256'])))
    for name,digest in CLIP['files_sha256'].items():
        if sha(root/name)!=digest:
            raise ValueError('Changed CLIP asset '+name)
    clip=CLIPModel.from_pretrained(root,local_files_only=True,use_safetensors=False,attn_implementation='eager').float().to(device).eval()
    processor=CLIPProcessor.from_pretrained(root,local_files_only=True,use_fast=False)
    ck=torch.load(weights,map_location='cpu',weights_only=False)
    for field in ('updates','source_sha256','parent_sha256'):
        if ck[field]!=RECEIPT[field]:
            raise ValueError('Wrong checkpoint lineage')
    model=CSAPatchCaptioner(len(words),ck['model']['core.nouns'],ck['model']['core.verbs'])
    model.load_state_dict(ck['model'],strict=True)
    del ck
    model.to(device).eval()

    def encode_text(strings):
        t=processor.tokenizer(strings,padding=True,truncation=False,return_tensors='pt').to(device)
        if t.input_ids.shape[1]>77:
            raise ValueError('CLIP input too long')
        return F.normalize(clip.get_text_features(**t).float(),dim=-1)

    with torch.inference_mode():
        missing=sorted({o['text'] for row in edited.values() for o in row['objects']}
                       | {r['text'] for row in edited.values() for r in row['relations']})
        missing=[t for t in missing if t not in texts]
        if missing:
            vectors=np.concatenate((vectors,encode_text(missing).cpu().numpy()))
            texts+=missing

        def generate(row,x,image_vector,detected):
            arrays=record_arrays(row,texts)
            prompt={k:v for k,v in arrays.items() if k not in ('object_text_id','relation_text_id')}
            for role in ('object','relation'):
                prompt[role+'_text']=vectors[arrays[role+'_text_id']]*arrays[role+'_valid'][:,None]
            prompt={k:torch.from_numpy(np.ascontiguousarray(v[None])).to(device) for k,v in prompt.items()}
            memory=model.encode(x,prompt)
            seq,scores=beam_search(model,memory,width=5,maximum=20)
            check_paths(model,memory,seq,scores)
            tokens=seq[0].cpu().tolist()
            caps=[text_from_ids(v,words) for v in tokens]
            sim=(2.5*(image_vector*encode_text(caps)).sum(-1).clamp_min(0)).cpu().tolist()
            candidates=[dict(ids=i,caption=c,logprob=l,clipscore=s) for i,c,l,s in zip(tokens,caps,scores[0].cpu().tolist(),sim,strict=True)]
            picked,_=rerank_multiscore(dict(candidates=candidates),detected,.4,.3)
            return dict(candidates=candidates,rank=picked['original_rank'],caption=picked['caption'])

        pairs=[]
        for cid,old in original.items():
            saved=SAVED[cid]
            path=Path(saved['image_path'])
            if sha(path)!=saved['image_sha256']:
                raise ValueError('Changed source image')
            with Image.open(path) as im:
                pixels=processor(images=im.convert('RGB'),return_tensors='pt')['pixel_values'].to(device)
            image_vector=F.normalize(clip.get_image_features(pixel_values=pixels).float(),dim=-1)
            x=torch.from_numpy(np.array(patches[index[cid]:index[cid]+1],dtype=np.float32)).to(device)
            control=generate(old,x,image_vector,saved['detected'])
            candidate=generate(edited[cid],x,image_vector,saved['detected'])
            historical=saved['beams'][saved['selected']]['caption']
            pairs.append(dict(coco_id=cid,control=control,fallback=candidate,historical=historical,
                              control_matches_history=control['caption']==historical))
            print(cid,control['caption'],' -> ',candidate['caption'],flush=True)
    (OUTPUT/'result.json').write_text(json.dumps(dict(images=17,proposals=29,applied=applied,blocked=blocked,
        new_clip_texts=missing,records=pairs,changed=sum(r['control']['caption']!=r['fallback']['caption'] for r in pairs),
        control_matches_history=sum(r['control_matches_history'] for r in pairs)),indent=2),encoding='utf-8')
    complete=True
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('result.json','notebook.log'):
            path=OUTPUT/name
            if path.is_file():z.write(path,name)
    print('SEND REVIEW:',review, 'complete:',complete,flush=True)
